In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import torch.optim as optim
from model import *
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
df = pd.read_csv('./dataset/train_processed.csv')
train_portion = int(len(df) * .75)
val_portion = int(len(df) * .8)
train_df = df.iloc[:train_portion]
val_df = df.iloc[val_portion:]
feature_cols = pd.read_table(
    'dataset/feature_columns.txt', header=None).iloc[:, 0].values
target_cols = ['target_short', 'target_medium', 'target_long']
assert train_df.isnull().sum().sum() == 0, "Training data contains missing values"
X_train = torch.tensor(train_df[feature_cols].values, dtype=torch.float32)
y_train = torch.tensor(train_df[target_cols].values, dtype=torch.float32)
X_val = torch.tensor(val_df[feature_cols].values, dtype=torch.float32)
y_val = torch.tensor(val_df[target_cols].values, dtype=torch.float32)
print(train_portion)
print(len(df) - val_portion)

In [ ]:
BATCH_SIZE = 16384

# =========================== MODELS ===========================
# 数据加载器
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True)  # 注意：时间序列通常不shuffle，这里为了演示
val_loader = DataLoader(val_dataset, batch_size=len(X_val), shuffle=False)
# 模型初始化
model = LSTMBaseLine(input_size=99, target_num=3,
                     hidden_dim=64, num_layers=2, dropout=0.2)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


In [ ]:
# ==================== HYPERPARAMETERS ====================

# 选择损失函数
criterion = WeightedMAELoss(weights=[0.5, 0.3, 0.2])  # 根据比赛重要性设置

# 优化器选择
optimizer = optim.AdamW(model.parameters(), lr=1e-3,
                       weight_decay=0.01)  # 推荐Adam
# optimizer = optim.RMSprop(model.parameters(), lr=1e-3, alpha=0.99)
# optimizer = optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)  # 收敛慢但可能更稳定

# 学习率调度器
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# ==================== TRAINING PROCESS AND BATCH SIZE ====================
num_epochs = 50
best_val_loss = float('inf')
best_epoch = -1

for epoch in range(num_epochs):
    # ========== 训练阶段 ==========
    model.train()
    train_loss = 0.0
    # hx = None

    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        
        # if hx is not None:
            # hx = (hx[0].detach(), hx[1].detach())
        
        output = model(data)  # 前向传播
        # output, hx = model(data, hx)  # 前向传播

        # 可以选择只取最后一个时间步的预测，或者用所有时间步
        # 方案A：用所有时间步的预测
        loss = criterion(output, target)

        loss.backward()

        # 梯度裁剪 - 防止梯度爆炸（对LSTM很重要）
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # ========== 验证阶段 ==========
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output= model(data)
            loss = criterion(output, target)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    # 学习率调整
    scheduler.step(avg_val_loss)

    # 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch
        torch.save(model.state_dict(), f'models/{model.__class__.__name__}_best_model.pth')

    print(
        f'Epoch {epoch+1}/{num_epochs}: Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')

print(f'训练完成，最佳验证损失: {best_val_loss:.6f}, 出现在第 {best_epoch+1} 个epoch。')

In [ ]:
import numpy as np
# 使用最佳模型进行完整的验证集验证，对每个target单独计算MAE并加权平均
model.load_state_dict(torch.load(f'models/{model.__class__.__name__}_best_model.pth'))
model.eval()

predictions = []
with torch.no_grad():
    hx = None
    for data, _ in val_loader:
        print(data.shape)
        data = data.to(device)
        output = model(data)
        predictions.append(output.cpu().numpy())
predictions = np.concatenate(predictions, axis=0)
predictions.shape

In [ ]:
y_val_np = y_val.numpy()
mae = np.mean(np.abs(predictions - y_val_np), axis=0)
print("各目标MAE:", mae)
weighted_mae = np.sum(mae * [0.5, 0.3, 0.2])
print("加权MAE:", weighted_mae)

In [ ]:

test_df = pd.read_csv('./dataset/test_processed.csv')
X_test = torch.tensor(test_df[feature_cols].values, dtype=torch.float32)
test_dataset = TensorDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

model.load_state_dict(torch.load(f'{model.__class__.__name__}_best_model.pth'))
model.eval()

predictions = []
with torch.no_grad():
    for data, *_ in test_loader:
        data = data.to(device)
        output = model(data)
        predictions.append(output.cpu().numpy())

In [ ]:
predictions = np.concatenate(predictions, axis=0)
predictions.shape

In [ ]:
pred_df = pd.DataFrame(predictions, columns=target_cols)
#将索引命名为id
pred_df.index.name = 'id'
pred_df.to_csv(f'results/{model.__class__.__name__}.csv')